# 06 · Deployment -- FastAPI + Streamlit + Hugging Face Spaces

This notebook covers the serving layer for the fraud detection model:

1. **FastAPI** -- a production REST API with `/health` and `/predict` endpoints,
   where every prediction includes the top-5 SHAP contributors
2. **Streamlit** -- an interactive dashboard with live sliders, colour-coded
   fraud probability gauge, SHAP waterfall, and threshold explorer
3. **Hugging Face Spaces** -- packaging the app as a self-contained deployment
   that runs without a local MLflow server

We can't run a persistent server inside a notebook cell, so we test the FastAPI
app by launching it as a subprocess, hitting it with `httpx`, then shutting it down.


In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, pickle, subprocess, time, textwrap
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR  = Path("../data")
MODELS_DIR = Path("../models")


## FastAPI -- `api/main.py`

The API lives at `api/main.py` in the project root.  Key design decisions:

| Decision | Rationale |
|---|---|
| Model loaded once at startup | Avoids per-request MLflow calls; cold-start is paid once |
| SHAP on scaled input | `TreeExplainer` runs on the booster directly after scaling, matching training |
| `Pipeline` artifact | Single `.predict_proba(X_raw)` call handles scaling internally |
| Engineered features auto-derived | Callers only send `Time`, `Amount`, V1-V28 |
| Pydantic validation | Invalid inputs rejected at the schema layer with a 422 response |

### Endpoints

```
GET  /health   →  model name, version, alias, feature count, threshold
POST /predict  →  fraud_probability, is_fraud, threshold_used, top_shap[5]
```

### Example request / response


In [2]:
example_fraud = {
    "Time": 75069, "Amount": 529.0,
    "V1": -2.3122, "V2": 1.9519, "V3": -1.6097, "V4": 3.9979,
    "V5": -0.5224, "V6": -1.4265, "V7": -2.5374, "V8": 0.8940,
    "V9": -0.2983, "V10": -3.5737, "V11": 1.3401, "V12": -4.2562,
    "V13": 0.0486, "V14": -5.7915, "V15": -0.3369, "V16": -3.0244,
    "V17": -5.5689, "V18": -1.0327, "V19": 0.5671, "V20": -0.0849,
    "V21": -0.2005, "V22": 0.3860, "V23": -0.0348, "V24": -0.0714,
    "V25": -0.1978, "V26": -0.3674, "V27": 0.1604, "V28": 0.1258,
}
example_legit = {
    "Time": 50000, "Amount": 12.5,
    "V1": 1.19, "V2": 0.26, "V3": 0.16, "V4": 0.45,
    "V5": -0.18, "V6": -0.35, "V7": 0.11, "V8": 0.08,
    "V9": -0.26, "V10": 0.07, "V11": -0.09, "V12": 0.14,
    "V13": -0.06, "V14": 0.23, "V15": 0.50, "V16": 0.11,
    "V17": -0.05, "V18": 0.12, "V19": 0.07, "V20": 0.02,
    "V21": -0.01, "V22": 0.05, "V23": -0.03, "V24": 0.01,
    "V25": 0.12,  "V26": 0.06, "V27": 0.01, "V28": 0.01,
}
print("Example fraud transaction (V14=-5.79 is a strong fraud signal):")
print(json.dumps({k: example_fraud[k] for k in ["Time","Amount","V14","V17","V12"]}, indent=2))


Example fraud transaction (V14=-5.79 is a strong fraud signal):
{
  "Time": 75069,
  "Amount": 529.0,
  "V14": -5.7915,
  "V17": -5.5689,
  "V12": -4.2562
}


## Live API Test

We start the FastAPI server as a subprocess, run a health check and two
prediction requests, then shut it down.


In [3]:
import sys, os
# Start the server from the project root
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--port", "8000", "--log-level", "error"],
    cwd=str(Path("..").resolve()),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(6)   # wait for startup
print(f"Server PID: {server.pid}")


Server PID: 16536


In [4]:
try:
    import httpx
except ImportError:
    import subprocess as _sp
    _sp.run([sys.executable, "-m", "pip", "install", "httpx", "-q"])
    import httpx

client = httpx.Client(base_url="http://localhost:8000", timeout=15)

health = client.get("/health").json()
print("GET /health")
print(json.dumps(health, indent=2))


GET /health
{
  "status": "ok",
  "model_name": "fraud-detector",
  "model_version": 1,
  "alias": "production",
  "n_features": 34,
  "threshold": 0.9848
}


In [5]:
print("POST /predict  (known fraud transaction)")
resp_fraud = client.post("/predict", json=example_fraud).json()
print(json.dumps(resp_fraud, indent=2))


POST /predict  (known fraud transaction)
{
  "fraud_probability": 0.999809,
  "is_fraud": true,
  "threshold_used": 0.9848,
  "top_shap": [
    {
      "feature": "V14",
      "value": -5.7915,
      "shap": 5.556429862976074
    },
    {
      "feature": "V17",
      "value": -5.5689,
      "shap": 1.6507573127746582
    },
    {
      "feature": "V10",
      "value": -3.5737,
      "shap": 1.1918483972549438
    },
    {
      "feature": "V12",
      "value": -4.2562,
      "shap": 1.1109837293624878
    },
    {
      "feature": "is_round_amount",
      "value": 1.0,
      "shap": -0.9656173586845398
    }
  ]
}


In [6]:
print("POST /predict  (benign transaction)")
resp_legit = client.post("/predict", json=example_legit).json()
print(json.dumps(resp_legit, indent=2))


POST /predict  (benign transaction)
{
  "fraud_probability": 0.004319,
  "is_fraud": false,
  "threshold_used": 0.9848,
  "top_shap": [
    {
      "feature": "V14",
      "value": 0.23,
      "shap": -2.8675193786621094
    },
    {
      "feature": "V12",
      "value": 0.14,
      "shap": -0.9683603048324585
    },
    {
      "feature": "amount_rolling_mean",
      "value": 12.5,
      "shap": -0.8375490307807922
    },
    {
      "feature": "V26",
      "value": 0.06,
      "shap": -0.4461115300655365
    },
    {
      "feature": "V10",
      "value": 0.07,
      "shap": -0.3504875600337982
    }
  ]
}


In [7]:
print(f"Fraud row   : p={resp_fraud['fraud_probability']:.6f}  "
      f"flagged={resp_fraud['is_fraud']}")
print(f"Legit row   : p={resp_legit['fraud_probability']:.6f}  "
      f"flagged={resp_legit['is_fraud']}")
print()
print("Top SHAP contributors for fraud transaction:")
for c in resp_fraud["top_shap"]:
    sign = "+" if c["shap"] > 0 else ""
    print(f"  {c['feature']:<22}  value={c['value']:>8.3f}  "
          f"SHAP={sign}{c['shap']:.4f}")
client.close()
server.terminate()
print("\nServer stopped.")


Fraud row   : p=0.999809  flagged=True
Legit row   : p=0.004319  flagged=False

Top SHAP contributors for fraud transaction:
  V14                     value=  -5.792  SHAP=+5.5564
  V17                     value=  -5.569  SHAP=+1.6508
  V10                     value=  -3.574  SHAP=+1.1918
  V12                     value=  -4.256  SHAP=+1.1110
  is_round_amount         value=   1.000  SHAP=-0.9656

Server stopped.


## Streamlit Dashboard -- `app/streamlit_app.py`

The dashboard is at `app/streamlit_app.py`.  It loads the registered MLflow model
at startup and then re-renders on every slider interaction.

### Key sections

| Section | Description |
|---|---|
| Sidebar | Sliders for Amount, Time, and top-5 SHAP features; threshold slider |
| Score card | Large colour-coded probability gauge (green/amber/red) |
| Feature table | Top-5 SHAP values for the current input |
| SHAP waterfall | Live horizontal bar chart -- updates on every slider move |
| Threshold explorer | Precision & Recall vs threshold on the held-out test set |

### To run locally

```bash
# From the fraud-detection/ directory:
streamlit run app/streamlit_app.py
# → http://localhost:8501
```


In [8]:
# Show the dashboard source (first 60 lines)
dashboard_src = Path("../app/streamlit_app.py").read_text(encoding="utf-8")
print(dashboard_src[:3000])
print("\n... [truncated -- see app/streamlit_app.py for full source]")


"""
app/streamlit_app.py — Interactive fraud-detection dashboard.

Run with:
    streamlit run app/streamlit_app.py

Sections
--------
1. Sidebar    — transaction input sliders (Amount + top-5 SHAP features)
2. Score card — large colour-coded fraud probability gauge
3. SHAP panel — live waterfall chart explaining the current prediction
4. Threshold  — interactive precision / recall tradeoff curve
"""

from __future__ import annotations

import sys
from pathlib import Path

# Make src/ importable when launched from the project root
sys.path.insert(0, str(Path(__file__).parent.parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mlflow.sklearn
import numpy as np
import pandas as pd
import shap
import streamlit as st
from mlflow import MlflowClient
from sklearn.metrics import precision_recall_curve

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="F

## Hugging Face Spaces Deployment

The `hf_space/` directory contains a self-contained version of the dashboard
that works without a local MLflow server -- the model is loaded from `model.pkl`
and the PR curve from `pr_curve.npz`.

### Files

```
hf_space/
├── app.py           ← Streamlit dashboard (pickle-backed, no MLflow needed)
├── model.pkl        ← Trained sklearn Pipeline  (1.0 MB)
├── pr_curve.npz     ← Pre-computed PR arrays    (586 KB)
├── requirements.txt ← Pinned dependencies
└── README.md        ← HF Spaces YAML header + docs
```

### Step-by-step deployment


In [9]:
# HF Spaces -- step-by-step deployment
steps = [
    '1. huggingface.co/new-space -> New Space -> SDK: Streamlit -> Create',
    '2. Upload hf_space/ files via Files -> Add file -> Upload files',
    '   (model.pkl is 1 MB -- no Git LFS required)',
    '3. HF installs requirements.txt and runs app.py automatically (~2-3 min)',
    '4. huggingface.co/spaces/YOUR_USERNAME/credit-card-fraud-detector',
]
for s in steps: print(s)

1. huggingface.co/new-space -> New Space -> SDK: Streamlit -> Create
2. Upload hf_space/ files via Files -> Add file -> Upload files
   (model.pkl is 1 MB -- no Git LFS required)
3. HF installs requirements.txt and runs app.py automatically (~2-3 min)
4. huggingface.co/spaces/YOUR_USERNAME/credit-card-fraud-detector


In [10]:
# Verify hf_space artifacts exist and are the right size
hf = Path("../hf_space")
for fname in ["app.py","model.pkl","pr_curve.npz","requirements.txt","README.md"]:
    p = hf / fname
    size = p.stat().st_size / 1024
    print(f"  {fname:<20}  {size:>8.1f} KB  {'OK' if p.exists() else 'MISSING'}")


  app.py                    14.3 KB  OK
  model.pkl               1029.2 KB  OK
  pr_curve.npz             585.7 KB  OK
  requirements.txt           0.1 KB  OK
  README.md                  1.7 KB  OK


## End-to-End Pipeline Summary

```
creditcard.csv
    │
    ├── 01_data_setup        stratified 80/20 split → train.csv / test.csv
    │
    ├── 02_eda_features      EDA plots → plots/
    │                        feature engineering (+6 cols)
    │                        SMOTE on train only → X/y_train_resampled.csv
    │
    ├── 03_modeling          6 classifiers compared by AUC-ROC / PR-AUC / F1
    │                        RandomizedSearchCV 50-iter on XGBoost
    │                        optimal threshold (precision >= 90%) → 0.9848
    │
    ├── 04_explainability    SHAP TreeExplainer
    │                        beeswarm / waterfall / force / dependence → plots/
    │
    ├── 05_mlflow            all 6 runs logged to fraud-detection experiment
    │                        best XGBoost → fraud-detector@production
    │
    └── 06_deployment        FastAPI /health + /predict (with SHAP)
                             Streamlit dashboard (live sliders, waterfall, PR explorer)
                             hf_space/ → Hugging Face Spaces
```

| Final Metric | Value |
|---|---|
| Best AUC-ROC | 0.9803 (Random Forest) |
| Tuned XGBoost AUC-ROC | 0.9779 |
| Precision at threshold | 90.7% |
| Recall at threshold | 79.6% |
| False positives (test set) | 8 |
| Missed fraud (test set) | 20 |
